<a href="https://colab.research.google.com/github/Nawat1124/wind-turbine-scada-anomaly-detection/blob/main/04b_lstm_threshold_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 04b Final experiment: LSTM threshold comparison

This notebook answers **RQ2** only.

It does **not** retrain the LSTM-AE. It loads the raw LSTM row scores saved by `04a_final_if_vs_lstm.ipynb`, then compares:

1. global q90;
2. per-asset q90;
3. KMeans-state q90.



In [1]:
# =========================================================
# 1. Load packages, data, and saved LSTM scores
# =========================================================

import os
import joblib
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

FREEZE_DIR = "/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v2"
RESULT_DIR = os.path.join(FREEZE_DIR, "results", "04_final_no_fl")
os.makedirs(RESULT_DIR, exist_ok=True)

WINDOW_FILE = os.path.join(FREEZE_DIR, "model_ready_simple", "windows_4h_stride2h_bundle.pkl")
EVENT_INFO_FILE = os.path.join(FREEZE_DIR, "event_info_v6.csv")
ROW_LINEAGE_FILE = os.path.join(FREEZE_DIR, "row_lineage_map.parquet")

LSTM_VAL_FILE = os.path.join(RESULT_DIR, "04a_lstm_validation_flat_row_scores.csv")
LSTM_PRED_FILE = os.path.join(RESULT_DIR, "04a_lstm_prediction_row_scores_raw_and_global_q90.csv")

assert os.path.exists(LSTM_VAL_FILE), "Run 04a first: missing validation row scores."
assert os.path.exists(LSTM_PRED_FILE), "Run 04a first: missing prediction row scores."

bundle = joblib.load(WINDOW_FILE)
df_model = bundle["df_model"].copy().reset_index(drop=True)

lstm_val_flat = pd.read_csv(LSTM_VAL_FILE)
lstm_rows_raw = pd.read_csv(LSTM_PRED_FILE)

Q = 0.90
NORMAL_STATUS_IDS = [0, 2]
CRITICALITY_THRESHOLD = 72
CARE_BETA = 0.5

print("df_model:", df_model.shape)
print("lstm_val_flat:", lstm_val_flat.shape)
print("lstm_rows_raw:", lstm_rows_raw.shape)


Mounted at /content/drive
df_model: (560060, 71)
lstm_val_flat: (81072, 4)
lstm_rows_raw: (72096, 76)


In [2]:
# =========================================================
# 2. Build official CARE event rows
# =========================================================

event_info = pd.read_csv(EVENT_INFO_FILE, sep=";")
if len(event_info.columns) == 1:
    event_info = pd.read_csv(EVENT_INFO_FILE)

row_lineage_map = pd.read_parquet(ROW_LINEAGE_FILE)

event_info["event_id"] = event_info["event_id"].astype(int)
event_info["asset_id"] = event_info["asset_id"].astype(int)
event_info["event_start_id"] = event_info["event_start_id"].astype(int)
event_info["event_end_id"] = event_info["event_end_id"].astype(int)
event_info["event_label"] = event_info["event_label"].astype(str).str.lower()

row_lineage_map["asset_id"] = row_lineage_map["asset_id"].astype(int)
row_lineage_map["source_session_id"] = row_lineage_map["source_session_id"].astype(str)
row_lineage_map["source_pos"] = row_lineage_map["source_pos"].astype(int)
row_lineage_map["physical_order_index"] = row_lineage_map["physical_order_index"].astype(int)
row_lineage_map["status_type_id"] = row_lineage_map["status_type_id"].astype(int)

parts = []

for _, ev in event_info.iterrows():
    event_id = int(ev["event_id"])
    asset_id = int(ev["asset_id"])
    source_file = f"{event_id}.csv"

    d = row_lineage_map[
        (row_lineage_map["asset_id"] == asset_id)
        & (row_lineage_map["source_session_id"] == source_file)
        & (row_lineage_map["source_pos"] >= int(ev["event_start_id"]))
        & (row_lineage_map["source_pos"] <= int(ev["event_end_id"]))
    ].copy()

    d["event_id"] = event_id
    d["event_label"] = ev["event_label"]
    d["event_description"] = ev.get("event_description", "")

    parts.append(d[[
        "event_id",
        "asset_id",
        "physical_order_index",
        "reconstructed_time",
        "status_type_id",
        "event_label",
        "event_description",
    ]])

care_event_frame = pd.concat(parts, ignore_index=True)
care_event_frame = (
    care_event_frame
    .drop_duplicates(["event_id", "asset_id", "physical_order_index"])
    .sort_values(["event_id", "physical_order_index"])
    .reset_index(drop=True)
)

print("care_event_frame:", care_event_frame.shape)


care_event_frame: (61616, 7)


In [3]:
# =========================================================
# 3. Small CARE helper functions
# =========================================================

def f_beta(tp, fp, fn, beta=0.5):
    beta2 = beta ** 2
    denom = (1 + beta2) * tp + beta2 * fn + fp
    if denom == 0:
        return 0.0
    return ((1 + beta2) * tp) / denom


def max_criticality(status, alarm):
    c = 0
    max_c = 0

    for s, a in zip(status, alarm):
        if int(s) in NORMAL_STATUS_IDS:
            if int(a) == 1:
                c += 1
            else:
                c = max(c - 1, 0)
        max_c = max(max_c, c)

    return max_c


def weighted_earliness(alarm):
    alarm = np.asarray(alarm).astype(int)
    n = len(alarm)
    if n == 0:
        return np.nan

    pos = np.linspace(0, 1, n)
    weights = np.where(pos <= 0.5, 1.0, 2.0 * (1.0 - pos))
    return float((weights * alarm).sum() / weights.sum())


def care_evaluate(row_table, alarm_col, model_name):
    pred = row_table[["event_id", "asset_id", "physical_order_index", alarm_col]].copy()
    pred[alarm_col] = pred[alarm_col].fillna(0).astype(int)

    pred = (
        pred
        .groupby(["event_id", "asset_id", "physical_order_index"], as_index=False)
        .agg(alarm=(alarm_col, "max"))
    )

    eval_rows = care_event_frame.merge(
        pred,
        on=["event_id", "asset_id", "physical_order_index"],
        how="left"
    )
    eval_rows["alarm"] = eval_rows["alarm"].fillna(0).astype(int)

    event_rows = []

    for event_id, d in eval_rows.groupby("event_id"):
        d = d.sort_values("physical_order_index")
        label = d["event_label"].iloc[0]
        asset_id = int(d["asset_id"].iloc[0])

        normal_rows = d[d["status_type_id"].isin(NORMAL_STATUS_IDS)]
        alarm = normal_rows["alarm"].astype(int).to_numpy()
        n = len(alarm)
        n_alarm = int(alarm.sum())

        coverage = np.nan
        accuracy = np.nan
        earliness = np.nan

        if label == "anomaly":
            coverage = f_beta(tp=n_alarm, fp=0, fn=n - n_alarm, beta=CARE_BETA)
            earliness = weighted_earliness(alarm)
        else:
            accuracy = (n - n_alarm) / n if n > 0 else np.nan

        max_c = max_criticality(d["status_type_id"].to_numpy(), d["alarm"].to_numpy())

        event_rows.append({
            "model": model_name,
            "event_id": int(event_id),
            "asset_id": asset_id,
            "event_label": label,
            "event_true_label": int(label == "anomaly"),
            "event_pred_label": int(max_c > CRITICALITY_THRESHOLD),
            "n_normal_status_rows": n,
            "n_alarm_rows": n_alarm,
            "alarm_rate_normal_status": n_alarm / n if n > 0 else np.nan,
            "coverage": coverage,
            "accuracy": accuracy,
            "weighted_earliness": earliness,
            "max_criticality": max_c,
            "event_description": d["event_description"].iloc[0],
        })

    detail = pd.DataFrame(event_rows)
    anom = detail[detail["event_label"] == "anomaly"]
    norm = detail[detail["event_label"] == "normal"]

    tp_event = int(((detail["event_true_label"] == 1) & (detail["event_pred_label"] == 1)).sum())
    fn_event = int(((detail["event_true_label"] == 1) & (detail["event_pred_label"] == 0)).sum())
    fp_event = int(((detail["event_true_label"] == 0) & (detail["event_pred_label"] == 1)).sum())

    reliability = f_beta(tp_event, fp_event, fn_event, beta=CARE_BETA)

    summary = pd.DataFrame([{
        "model": model_name,
        "TP/FN/FP": f"{tp_event}/{fn_event}/{fp_event}",
        "mean_coverage": anom["coverage"].mean(),
        "mean_earliness": anom["weighted_earliness"].mean(),
        "event_reliability": reliability,
        "mean_normal_accuracy": norm["accuracy"].mean(),
        "anomaly_alarm_rate": anom["alarm_rate_normal_status"].mean(),
        "normal_alarm_rate": norm["alarm_rate_normal_status"].mean(),
    }])

    summary["CARE_score"] = (
        summary["mean_coverage"]
        + summary["mean_earliness"]
        + summary["event_reliability"]
        + 2 * summary["mean_normal_accuracy"]
    ) / 5

    if detail["event_pred_label"].sum() == 0:
        summary["CARE_score"] = 0.0

    return detail, summary, eval_rows


In [4]:
# =========================================================
# 4. Threshold 1: global q90
# =========================================================

global_q90 = float(lstm_val_flat["lstm_row_score"].quantile(Q))

rows_global = lstm_rows_raw.copy()
rows_global["lstm_alarm_global_q90"] = (rows_global["lstm_row_score"] > global_q90).astype(int)

print("Global q90:", global_q90)
print("Global q90 alarm rate:", rows_global["lstm_alarm_global_q90"].mean())


Global q90: 0.14210482090711593
Global q90 alarm rate: 0.25969540612516645


In [5]:
# =========================================================
# 5. Threshold 2: per-asset q90
# =========================================================

asset_thresholds = (
    lstm_val_flat
    .groupby("asset_id")["lstm_row_score"]
    .quantile(Q)
    .to_dict()
)

asset_threshold_table = (
    lstm_val_flat
    .groupby("asset_id")
    .agg(
        n_val_timestep_errors=("lstm_row_score", "size"),
        threshold=("lstm_row_score", lambda x: x.quantile(Q)),
        mean_score=("lstm_row_score", "mean"),
        max_score=("lstm_row_score", "max"),
    )
    .reset_index()
    .sort_values("asset_id")
)

rows_asset = lstm_rows_raw.copy()
rows_asset["threshold"] = rows_asset["asset_id"].map(asset_thresholds)
rows_asset["lstm_alarm_per_asset_q90"] = (rows_asset["lstm_row_score"] > rows_asset["threshold"]).astype(int)

print(asset_threshold_table.to_string(index=False))
print("Per-asset q90 alarm rate:", rows_asset["lstm_alarm_per_asset_q90"].mean())

asset_threshold_table.to_csv(os.path.join(RESULT_DIR, "04b_lstm_per_asset_q90_thresholds.csv"), index=False)


 asset_id  n_val_timestep_errors  threshold  mean_score  max_score
        0                   8880   0.129434    0.069312   0.776961
        2                   7944   0.179656    0.098949   3.872358
        5                   8592   0.146750    0.075635   1.374812
        6                  10848   0.123066    0.066008   1.186780
        7                   9096   0.136250    0.078685   2.927938
       11                   9096   0.141207    0.074405   1.118289
       12                   8856   0.169473    0.088723   2.927751
       13                   9264   0.131538    0.069586   1.281309
       14                   8496   0.122697    0.063484   0.919111
Per-asset q90 alarm rate: 0.25317632046160676


In [6]:
# =========================================================
# 6. Threshold 3: KMeans-state q90
# =========================================================

kmeans_thresholds = (
    lstm_val_flat
    .groupby("kmeans_state")["lstm_row_score"]
    .quantile(Q)
    .to_dict()
)

kmeans_threshold_table = (
    lstm_val_flat
    .groupby("kmeans_state")
    .agg(
        n_val_timestep_errors=("lstm_row_score", "size"),
        threshold=("lstm_row_score", lambda x: x.quantile(Q)),
        mean_score=("lstm_row_score", "mean"),
        max_score=("lstm_row_score", "max"),
    )
    .reset_index()
    .sort_values("kmeans_state")
)

rows_kmeans = lstm_rows_raw.copy()
rows_kmeans["threshold"] = rows_kmeans["kmeans_state"].map(kmeans_thresholds)
rows_kmeans["lstm_alarm_kmeans_state_q90"] = (rows_kmeans["lstm_row_score"] > rows_kmeans["threshold"]).astype(int)

print(kmeans_threshold_table.to_string(index=False))
print("KMeans-state q90 alarm rate:", rows_kmeans["lstm_alarm_kmeans_state_q90"].mean())

kmeans_threshold_table.to_csv(os.path.join(RESULT_DIR, "04b_lstm_kmeans_state_q90_thresholds.csv"), index=False)


 kmeans_state  n_val_timestep_errors  threshold  mean_score  max_score
            0                  13610   0.127014    0.066750   3.358231
            1                  15810   0.115603    0.068236   3.872358
            2                  18483   0.115170    0.063682   2.927938
            3                   7666   0.189959    0.102657   1.799985
            4                  15499   0.158808    0.084985   1.663125
            5                  10004   0.165823    0.086090   1.717049
KMeans-state q90 alarm rate: 0.2572403462050599


In [7]:
# =========================================================
# 7. CARE evaluation for all three threshold strategies
# =========================================================

global_detail, global_summary, _ = care_evaluate(
    rows_global,
    alarm_col="lstm_alarm_global_q90",
    model_name="LSTM-AE global q90",
)

asset_detail, asset_summary, _ = care_evaluate(
    rows_asset,
    alarm_col="lstm_alarm_per_asset_q90",
    model_name="LSTM-AE per-asset q90",
)

kmeans_detail, kmeans_summary, _ = care_evaluate(
    rows_kmeans,
    alarm_col="lstm_alarm_kmeans_state_q90",
    model_name="LSTM-AE KMeans-state q90",
)

rq2_table = pd.concat([global_summary, asset_summary, kmeans_summary], ignore_index=True)
rq2_table = rq2_table[[
    "model",
    "TP/FN/FP",
    "mean_coverage",
    "mean_earliness",
    "mean_normal_accuracy",
    "anomaly_alarm_rate",
    "normal_alarm_rate",
    "CARE_score",
]]

threshold_event_detail = pd.concat([global_detail, asset_detail, kmeans_detail], ignore_index=True)

print("RQ2: LSTM threshold comparison")
display(rq2_table)

print("Event detail")
display(threshold_event_detail.sort_values(["model", "event_id"]))

rq2_table.to_csv(os.path.join(RESULT_DIR, "04b_RQ2_lstm_threshold_comparison.csv"), index=False)
threshold_event_detail.to_csv(os.path.join(RESULT_DIR, "04b_RQ2_lstm_threshold_event_detail.csv"), index=False)

rows_global.to_csv(os.path.join(RESULT_DIR, "04b_lstm_rows_global_q90.csv"), index=False)
rows_asset.to_csv(os.path.join(RESULT_DIR, "04b_lstm_rows_per_asset_q90.csv"), index=False)
rows_kmeans.to_csv(os.path.join(RESULT_DIR, "04b_lstm_rows_kmeans_state_q90.csv"), index=False)

print("Saved to:", RESULT_DIR)


RQ2: LSTM threshold comparison


,model,TP/FN/FP,mean_coverage,mean_earliness,mean_normal_accuracy,anomaly_alarm_rate,normal_alarm_rate,CARE_score
0,LSTM-AE global q90,4/2/1,0.569469,0.215230,0.819823,0.217315,0.180177,0.638715
1,LSTM-AE per-asset q90,4/2/2,0.590150,0.227639,0.809284,0.227312,0.190716,0.620605
2,LSTM-AE KMeans-state q90,4/2/0,0.580262,0.220747,0.826177,0.225276,0.173823,0.672491


Event detail


,model,event_id,asset_id,event_label,event_true_label,event_pred_label,n_normal_status_rows,n_alarm_rows,alarm_rate_normal_status,coverage,accuracy,weighted_earliness,max_criticality,event_description
30,LSTM-AE KMeans-state q90,2,13,normal,0,0,1872,214,0.114316,NaN,0.885684,NaN,21,NaN
31,LSTM-AE KMeans-state q90,7,13,anomaly,1,1,4210,898,0.213302,0.575493,NaN,0.210509,143,high temperature in transformer cell
32,LSTM-AE KMeans-state q90,19,11,anomaly,1,1,2855,809,0.283363,0.664095,NaN,0.279166,125,high temperature in transformer cell
33,LSTM-AE KMeans-state q90,21,0,normal,0,0,1269,261,0.205674,NaN,0.794326,NaN,37,NaN
34,LSTM-AE KMeans-state q90,23,6,normal,0,0,1353,186,0.137472,NaN,0.862528,NaN,18,NaN
35,LSTM-AE KMeans-state q90,27,7,anomaly,1,0,8404,1568,0.186578,0.534206,NaN,0.196249,58,Turbine is stopped due to a main bearing damage
36,LSTM-AE KMeans-state q90,34,14,anomaly,1,0,3046,386,0.126724,0.420479,NaN,0.117263,36,high temperature in transformer cell
37,LSTM-AE KMeans-state q90,52,14,normal,0,0,2012,315,0.156561,NaN,0.843439,NaN,24,NaN
38,LSTM-AE KMeans-state q90,53,6,anomaly,1,1,5711,1242,0.217475,0.581515,NaN,0.238155,114,Rotor Bearing 2 - Damage
39,LSTM-AE KMeans-state q90,74,11,normal,0,0,1867,456,0.244242,NaN,0.755758,NaN,39,NaN


Saved to: /content/drive/MyDrive/windfarmB_reconstructed_observed_db_v2/results/04_final_no_fl
